In [1]:
import joblib
import os
import pandas as pd
import numpy as np
import random
import pickle
import hashlib
os.chdir('Resources/')

In [2]:
model = joblib.load('6_random_forest_model.joblib')
scaler = joblib.load('6_scaler.joblib')

data = random.randint(1, 4047)

In [3]:
df_h = pd.read_csv('4_Hashed_Data.csv')

X_new_h = df_h.iloc[data, :-1]  
X_new_h = np.array(X_new_h).reshape(1, -1)

X_new_h

array([[  16531176,  692011033, 2958827034, 1331294600,  275479829,
        3635578653, 4255392980,  742789580, 3557566171, 4149563607,
        4025547809]], dtype=int64)

In [4]:
X_new_scaled_h = scaler.transform(X_new_h)
pred_new_h = model.predict(X_new_scaled_h)

C:\Users\USER\AppData\Roaming\Python\Python311\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [5]:
def power(a, b, c):
    x = 1
    y = a
    while b > 0:
        if b % 2 != 0:
            x = (x * y) % c
        y = (y * y) % c
        b = int(b / 2)
    return x % c

In [6]:
def encrypt(value, q, h, g, fixed_key):
    s = power(h, fixed_key, q)
    p = power(g, fixed_key, q)
    return s * value, p

In [7]:
public_key_file_path = '3_Public_Key.pkl'

with open(public_key_file_path, 'rb') as public_key_file:
    column_keys = pickle.load(public_key_file)

In [8]:
df_s = pd.read_csv('1_Structured_Data.csv')

X_new_s = df_s.iloc[data, :-1]  
X_new_s

Age                                   49
Sex                                 Male
ChestPainType           Non-anginal Pain
RestingBloodPressure               115.0
Cholesterol                        265.0
FastingBloodSugar                     No
RestingECG                        Normal
MaximumHeartRate                   175.0
ExerciseAngina                        No
Oldpeak                              0.0
ST_Slope                            Flat
Name: 135, dtype: object

In [9]:
encrypted_row = {}

for column, value in X_new_s.items():
    q, h, g, fixed_key = column_keys[column]
    if isinstance(value, str):
        numeric_value = sum(ord(char) for char in value)
        encrypted_value, _ = encrypt(numeric_value, q, h, g, fixed_key)
    else:
        encrypted_value, _ = encrypt(float(value), q, h, g, fixed_key)
    encrypted_row[column] = encrypted_value

encrypted_df_s = pd.DataFrame([encrypted_row])

encrypted_df_s

,Age,Sex,ChestPainType,RestingBloodPressure,Cholesterol,FastingBloodSugar,RestingECG,MaximumHeartRate,ExerciseAngina,Oldpeak,ST_Slope
0,3.334288e+51,1350058023403595324343430093229461786520939609...,1867343179792106420018046177683851717368955346...,3.477985e+50,1.399360e+51,5560939900884587107690043723018959399006567134...,1876246555722014784023520193931757633827432838...,4.138583e+50,1126890232113142632220902390984674149293376678...,0.0,4509645400611199399377702071823246891155647968...


In [10]:
def hash_value(value):
    hash_object = hashlib.sha256(str(value).encode())
    hash_hex = hash_object.hexdigest()
    hash_int = int(hash_hex, 16) % (2**32)
    return hash_int

In [11]:
hashed_s = encrypted_df_s.applymap(hash_value)
hashed_s

C:\Users\USER\AppData\Local\Temp\ipykernel_22152\3352419567.py:1: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  hashed_s = encrypted_df_s.applymap(hash_value)


,Age,Sex,ChestPainType,RestingBloodPressure,Cholesterol,FastingBloodSugar,RestingECG,MaximumHeartRate,ExerciseAngina,Oldpeak,ST_Slope
0,16531176,692011033,2958827034,1331294600,275479829,3635578653,4255392980,742789580,3557566171,4149563607,4025547809


In [12]:
X_new_scaled_s = scaler.transform(hashed_s)
pred_new_s = model.predict(X_new_scaled_s)

In [13]:
print("Predicted label for:", data+2, "th data:", pred_new_h[0])
print("Predicted label for:", data+2, "th data:", pred_new_s[0])

Predicted label for: 137 th data: 1
Predicted label for: 137 th data: 1
